# XML, HTML, and Web Scraping

JSON and XML are two different ways to represent hierarchical data. Which one is better? There are lots of articles online which discuss similarities and differences between JSON and XML and their advantages and disadvantages. Both formats are still in current usage, so it is good to be familiar with both. However, JSON is more common, so we'll focus on working with JSON representations of hierarchical data.

The reading covered an example of using Beautiful Soup to parse XML. Rather than doing another example XML now, we'll skip straight to scraping HTML from a webpage. Both HTML and XML can be parsed in a similar way with Beautiful Soup.

In [30]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
#jason is simpler, has key{};  human eyes easy
#xml has tags<>;  complex but robust

## Scraping an HTML table with Beautiful Soup

Open the URL https://en.wikipedia.org/wiki/List_of_United_States_cities_by_population and scroll down until you see a table of the cities in the U.S. with population over 100,000 (as of Jul 1, 2022). We'll use Beautiful Soup to scrape information from this table.

Read in the HTML from the ULR using the `requests` library.

In [31]:
#pandas: for data frames
#requests: for downloading web page HTML
#BeautifulSoup: for parsing HTML/XML
import requests
URL = "https://en.wikipedia.org/wiki/List_of_United_States_cities_by_population"
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

response = requests.get(URL, headers=HEADERS)
response

<Response [200]>

Use Beautiful Soup to parse this string into a tree called `soup`

In [32]:
from bs4 import BeautifulSoup
soup = BeautifulSoup(response.content, 'html.parser')


To find an HTML tag corresponding to a specific element on a webpage, right-click on it and choose "Inspect element". Go to the cities table Wikipedia page and do this now.

You should find that the cities table on the Wikipedia page corresponds to the element

```
<table class="wikitable sortable jquery-tablesorter" style="text-align:center">
```

There are many `<table>` tags on the page.

In [33]:
#find all table elements in the HTML
len(soup.find_all("table"))

10

In [34]:
#print(soup.find_all("table"))

We can use attributes like `class=` and `style=` to narrow down the list.

In [35]:
len(soup.find_all("table",
                  attrs={
                      "class": "wikitable sortable sort-under col1left col2center",
                      "style": "text-align:right"}
                  ))

1

At this point, you can manually inspect the tables on the webpage to find that the one we want is the first one (see `[0]` below). We'll store this as `table`.

In [36]:
table = soup.find_all("table",
                  attrs={
                      "class": "wikitable sortable sort-under col1left col2center", #changed name to #wikitable sortable sort-under col1left col2center
                      "style": "text-align:right"} #changed style to alin:right from center(nav's notebook)
                  )[0]


**Now you will write code to scrape the information in `table` to create a Pandas data frame with one row for each city and columns for: city, state, population (2022 estimate), and 2020 land area (sq mi).** Refer to the Notes/suggestions below as you write your code. A few Hints are provided further down, but try coding first before looking at the hints.

Notes/suggestions:

- Use as a guide the code from the reading that produced the data frame of Statistics faculty
- Inspect the page source as you write your code
- You will need to write a loop to get the information for all cities, but you might want to try just scraping the info for New York first
- You will need to pull the text from the tag. If `.text` returns text with "\n" at the end, try `.get_text(strip = True)` instead of `.text`
- Don't forget to convert to a Pandas Data Frame; it should have 333 rows and 4 columns
- The goal of this exercise is just to create the Data Frame. If you were going to use it --- e.g., what is the population density for all cities in CA? --- then you would need to clean the data first (to clean strings and convert to quantitative). (You can use Beautiful Soup to do some of the cleaning for you, but that goes beyond our scope.)

In [37]:
def clean_cell(tag):
    for sup in tag.find_all("sup"):
        sup.decompose()
    return tag.get_text(strip=True)

In [38]:
rows = []
for tr in table.find_all("tr")[1:]:
    tds = tr.find_all("td")
    if len(tds) < 6:
        continue

    # After the rank , the  order is:
    # 0 City | 1 State | 2 2022 estimate | 3 2020 census | 4 Change | 5 2020 land area (sq mi) | ...
    city  = clean_cell(tds[0])
    state = clean_cell(tds[1])
    pop22 = clean_cell(tds[2])
    area  = clean_cell(tds[5])

    rows.append({
        "city": city,
        "state": state,
        "population_2022_estimate": pop22,
        "land_area_sq_mi": area
    })



In [39]:
df_cities = pd.DataFrame(rows)
df_cities.shape, df_cities.head()

((25, 4),
              city state population_2022_estimate land_area_sq_mi
 0       Allegheny    PA                       NA            1907
 1        Brooklyn    NY                       NA            1898
 2          Camden    NJ                   71,749            1950
 3          Canton    OH                   69,211            1950
 4  Citrus Heights    CA                   86,909            1990)

Hints:

- Each city is a row in the table; find all the `<tr>` tags to find all the cities
- Look for the `<td>` tag to see table entries within a row
- The rank column is represented by `<th>` tags, rather than `<td>` tags. So within a row, the first (that is, `[0]`) `<td>` tag corresponds to the city name.

## Aside: Scraping an HTML table with Pandas



The Pandas command `read_html` can be used to scrape information from an HTML table on a webpage.

We can call `read_html` on the URL.

In [40]:
#pd.read_html("https://en.wikipedia.org/wiki/List_of_United_States_cities_by_population")

However, this scrapes all the tables on the webpage, not just the one we want. As with Beautiful Soup, we can narrow the search by specifying the table attributes.

In [41]:
#pd.read_html("https://en.wikipedia.org/wiki/List_of_United_States_cities_by_population", attrs = {'class': 'wikitable sortable', "style": "text-align:center"})

This still returns 3 tables. As we remarked above, the table that we want is the first one (see `[0]` below).

In [42]:
#df_cities2 = pd.read_html("https://en.wikipedia.org/wiki/List_of_United_States_cities_by_population", attrs = {'class': 'wikitable sortable', "style": "text-align:center"})[0]
#df_cities2

Wait, that seemed much easier than using Beautiful Soup, and it returned a data frame, and we even got for free some formatting like removing the commas from the population! Why didn't we just use `read_html` in the first place? It's true the `read_html` works well when scraping information from an HTML *table*. Unfortunately, you often want to scrape information from a webpage that isn't conveniently stored in an HTML table, in which case `read_html` won't work. (It only searches for `<table>`, `<th>`, `<tr>`, and `<td>` tags, but there are many other HTML tags.) Though Beautiful Soup is not as simple as `read_html`, it is more flexible and thus more widely applicable.

## Scraping information that is NOT in a `<table>` with Beautiful Soup

The Cal Poly course catalog http://catalog.calpoly.edu/collegesandprograms/collegeofsciencemathematics/statistics/#courseinventory contains a list of courses offered by the Statistics department. **You will scrape this website to obtain a Pandas data frame with one row for each DATA or STAT course and two columns: course name and number (e.g, DATA 301. Introduction to Data Science) and term typically offered (e.g., Term Typically Offered: F, W, SP).**

Note: Pandas `read_html` is not help here since the courses are not stored in a `<table>.`

In [43]:
pd.read_html("http://catalog.calpoly.edu/collegesandprograms/collegeofsciencemathematics/statistics/#courseinventory")

[                                        Program name   Program type
 0                              Actuarial Preparation          Minor
 1  Cross Disciplinary Studies Minor in Bioinforma...          Minor
 2   Cross Disciplinary Studies Minor in Data Science          Minor
 3                                         Statistics  BS, MS, Minor]


Notes/suggestions:


- Inspect the page source as you write your code
- The courses are not stored in a `<table>`. How are they stored?
- You will need to write a loop to get the information for all courses, but you might want to try just scraping the info for DATA 100 first
- What kind of tag is the course name stored in? What is the `class` of the tag?
- What kind of tag is the quarter(s) the course is offered stored in? What is the `class` of the tag? Is this the only tag of this type with the class? How will you get the one you want?
- You don't have to remove the number of units (e.g., 4 units) from the course name and number, but you can try it if you want
- You will need to pull the text from the tag. If `.text` returns text with "\n" at the end, try `get_text(strip = True)` instead of `text`
- Don't forget to convert to a Pandas Data Frame; it should have 74 rows and 2 columns
- The goal of this exercise is just to create the Data Frame. If you were going to use it then you might need to clean the data first. (You can use Beautiful Soup to do some of the cleaning for you, but that goes beyond our scope.)



In [44]:
CATALOG_URL = "http://catalog.calpoly.edu/collegesandprograms/collegeofsciencemathematics/statistics/#courseinventory"
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

resp = requests.get(CATALOG_URL, headers=HEADERS)
soup = BeautifulSoup(resp.content, "html.parser")



In [45]:
courses = []
for block in soup.find_all("div", {"class": "courseblock"}):
    title_tag = block.find("p", {"class": "courseblocktitle"})
    if not title_tag:
        continue

    title_text = title_tag.get_text(" ", strip=True)
    if not (title_text.startswith("STAT") or title_text.startswith("DATA")):
        continue

    # remove "(X units)" span if present
    for span in title_tag.find_all("span"):
        span.extract()
    course_name = title_tag.get_text(" ", strip=True)

    # first noindent line is typically "Term Typically Offered: ..."
    term_tag = block.find("p", {"class": "noindent"})
    term_text = term_tag.get_text(" ", strip=True) if term_tag else ""

    # if not the right line, search all noindent p's for the one that starts with it
    if not term_text.lower().startswith("term typically offered"):
        for p in block.find_all("p", {"class": "noindent"}):
            t = p.get_text(" ", strip=True)
            if t.lower().startswith("term typically offered"):
                term_text = t
                break

    courses.append({"course": course_name, "term_typically_offered": term_text})


In [46]:
df_courses = pd.DataFrame(courses)
df_courses.shape, df_courses.head()

((74, 2),
                                               course  \
 0                  DATA 100. Data Science for All I.   
 1            DATA 301. Introduction to Data Science.   
 2         DATA 401. Data Science Process and Ethics.   
 3  DATA 402. Mathematical Foundations of Data Sci...   
 4        DATA 403. Data Science Projects Laboratory.   
 
              term_typically_offered  
 0  Term Typically Offered: F, W, SP  
 1  Term Typically Offered: F, W, SP  
 2         Term Typically Offered: F  
 3         Term Typically Offered: F  
 4         Term Typically Offered: F  )

Hints:

- Each course is represented by a `<div>` with `class=courseblock`, so you can find all the courses with `soup.find_all("div", {"class": "courseblock"})`
- The course name is in a `<p>` tag with `class=courseblocktitle`, inside a `<strong>` tag. (Though I don't think we need to find the strong tag here.)
- The term typically offered is in `<p>` tag with `class=noindent`. However, there are several tags with this class; term typically offered is the first one.
- If you want to use Beautiful Soup to remove the course units (e.g., 4 units), find the `<span>` tag within the course name tag and `.extract()` this span tag